# 🌍 Earthquake Magnitude & Depth Prediction — Random Forest

**Dataset:** USGS Significant Earthquakes (1965–2016)
**Goal:** Predict `Magnitude` and `Depth` from `Timestamp`, `Latitude`, and `Longitude`.
**Model:** `RandomForestRegressor` (scikit-learn), tuned via `GridSearchCV`.

---

## Scientific Motivation

Earthquake magnitude and depth are governed by complex, nonlinear tectonic processes — plate
boundary type, fault geometry, regional stress accumulation — that vary systematically by
location and, more weakly, by time. Unlike a parametric geophysical model, a Random Forest
requires no explicit assumptions about fault mechanics: it learns nonlinear interactions
between latitude, longitude, and time directly from ~5 decades of globally catalogued seismic
events. This notebook treats that catalog as a regression benchmark and establishes a strong
tabular baseline against which the signal-based deep learning approaches in Notebooks 2–3 are
later compared.

### Notebook outline
1. Imports & configuration
2. Load & preprocess data
3. Exploratory data analysis
4. Train / test split (80 / 20)
5. Hyperparameter tuning (`GridSearchCV`)
6. Held-out evaluation
7. Feature importance
8. Persist the trained model
9. Result summary


## 1 · Imports & Configuration

In [ ]:
import os
import sys

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

sys.path.append(os.path.abspath(os.path.join("..", "src")))
from config import (
    USGS_RAW_PATH, USGS_PROCESSED_PATH, USGS_FEATURE_COLUMNS,
    USGS_TARGET_COLUMNS, USGS_TEST_SIZE, RF_PARAM_GRID, RF_CV_FOLDS,
    RF_SCORING, RANDOM_STATE, MODELS_DIR, FIGURES_DIR,
    PLOT_COLORS, PLOT_DPI, PLOT_STYLE,
)
from logging_utils import console, get_logger, metrics_table, section
from preprocessing import load_and_clean_usgs

logger = get_logger(__name__)

# Shared dark, high-contrast publication theme (identical to src/train_random_forest.py)
plt.style.use(PLOT_STYLE)
sns.set_style("darkgrid", {"axes.facecolor": "#1b1f27", "grid.color": PLOT_COLORS["grid"]})
plt.rcParams.update({
    "figure.dpi": PLOT_DPI,
    "savefig.dpi": PLOT_DPI,
    "axes.edgecolor": PLOT_COLORS["text"],
    "axes.labelcolor": PLOT_COLORS["text"],
    "text.color": PLOT_COLORS["text"],
    "xtick.color": PLOT_COLORS["text"],
    "ytick.color": PLOT_COLORS["text"],
    "font.size": 11,
})

console.print(f"[bold]Random seed:[/bold] {RANDOM_STATE}")

## 2 · Load & Preprocess Data

The raw USGS catalog stores `Date` and `Time` as separate string columns. `preprocessing.load_and_clean_usgs`
merges these into a single numeric `Timestamp` (Unix epoch seconds) — preserving genuine temporal structure
(aftershock sequences, regional seismic cycles) as a feature a tree-based model can split on directly — then
filters the dataframe to the five columns the model actually needs.

In [ ]:
section("Load & Preprocess USGS Data")

# Expects the Kaggle "Significant Earthquakes, 1965-2016" database.csv at data/raw/database.csv
df = load_and_clean_usgs(USGS_RAW_PATH)

os.makedirs(os.path.dirname(USGS_PROCESSED_PATH), exist_ok=True)
df.to_csv(USGS_PROCESSED_PATH, index=False)
logger.info("Saved processed dataset to '%s' (%d rows).", USGS_PROCESSED_PATH, len(df))

console.print(f"[green]Loaded[/green] {len(df):,} clean rows.")
df.head()

In [ ]:
df.describe()

## 3 · Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["Magnitude"], bins=40, kde=True, ax=axes[0], color=PLOT_COLORS["primary"])
axes[0].set_title("Distribution of Earthquake Magnitude", fontweight="bold")
axes[0].set_xlabel("Magnitude (moment magnitude scale)")
axes[0].set_ylabel("Count")
axes[0].grid(alpha=0.3)

sns.histplot(df["Depth"], bins=40, kde=True, ax=axes[1], color=PLOT_COLORS["secondary"])
axes[1].set_title("Distribution of Earthquake Depth", fontweight="bold")
axes[1].set_xlabel("Depth (km)")
axes[1].set_ylabel("Count")
axes[1].grid(alpha=0.3)

fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "rf_eda_distributions.png"), dpi=PLOT_DPI, bbox_inches="tight")
plt.show()

**Reading the distributions:** Magnitude follows an approximately log-normal shape consistent with the
Gutenberg–Richter relation (small events vastly outnumber large ones, even within this "significant events
only" catalog). Depth is heavily right-skewed, dominated by shallow crustal events (<70 km), with a long tail
of intermediate- and deep-focus events associated with subducting slabs.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(
    df["Longitude"], df["Latitude"], c=df["Magnitude"],
    cmap="inferno", s=6, alpha=0.6, edgecolors="none",
)
cbar = fig.colorbar(sc, ax=ax, label="Magnitude")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Global Distribution of Significant Earthquakes (1965-2016)", fontweight="bold")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "rf_global_distribution.png"), dpi=PLOT_DPI, bbox_inches="tight")
plt.show()

The spatial scatter traces the world's major plate boundaries almost perfectly — the Pacific Ring of Fire,
the mid-Atlantic ridge, and the Alpide belt are all visible without any explicit geographic labeling. This is
the structural signal `Latitude`/`Longitude` give the Random Forest to exploit.

## 4 · Feature / Target Split & Train–Test Split (80 / 20)

In [ ]:
X = df[USGS_FEATURE_COLUMNS]
y = df[USGS_TARGET_COLUMNS]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=USGS_TEST_SIZE, random_state=RANDOM_STATE
)

console.print(f"Train: [bold]{X_train.shape}[/bold]  |  Test: [bold]{X_test.shape}[/bold]")

## 5 · Hyperparameter Tuning — `GridSearchCV`

We tune `n_estimators` over the project-wide grid defined in `config.RF_PARAM_GRID`, using
`config.RF_CV_FOLDS`-fold cross-validation and optimizing for `config.RF_SCORING` (R²).

In [ ]:
section("Hyperparameter Tuning (GridSearchCV)")
console.print(f"Searching n_estimators over {RF_PARAM_GRID['n_estimators']} ({RF_CV_FOLDS}-fold CV)...")

base_model = RandomForestRegressor(random_state=RANDOM_STATE)

grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=RF_PARAM_GRID,
    cv=RF_CV_FOLDS,
    scoring=RF_SCORING,
    n_jobs=-1,
    verbose=0,
)

grid_search.fit(X_train, y_train)

console.print(
    f"[bold green]Best n_estimators:[/bold green] {grid_search.best_params_['n_estimators']}  "
    f"|  [bold green]Best CV R2:[/bold green] {grid_search.best_score_:.4f}"
)

In [ ]:
results = pd.DataFrame(grid_search.cv_results_)
results = results[["param_n_estimators", "mean_test_score", "std_test_score"]]
results = results.sort_values("param_n_estimators")

fig, ax = plt.subplots(figsize=(9, 5))
ax.errorbar(
    results["param_n_estimators"], results["mean_test_score"],
    yerr=results["std_test_score"], marker="o", capsize=4,
    color=PLOT_COLORS["primary"], ecolor=PLOT_COLORS["secondary"],
    linewidth=2, markersize=7,
)
ax.set_xlabel("n_estimators")
ax.set_ylabel("Mean CV $R^2$ Score")
ax.set_title("GridSearchCV: $R^2$ vs. n_estimators", fontweight="bold")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "rf_grid_search_curve.png"), dpi=PLOT_DPI, bbox_inches="tight")
plt.show()

results

## 6 · Final Model Evaluation on Held-Out Test Set

In [ ]:
section("Evaluation on Held-Out Test Set")

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

metrics = {
    "r2": r2_score(y_test, y_pred),
    "mae": mean_absolute_error(y_test, y_pred),
    "rmse": float(np.sqrt(mean_squared_error(y_test, y_pred))),
}
console.print(metrics_table("Random Forest -- Test Set Metrics", metrics))

In [ ]:
# Per-target R2 / MAE breakdown (Magnitude vs. Depth)
y_pred_df = pd.DataFrame(y_pred, columns=USGS_TARGET_COLUMNS, index=y_test.index)

for col in USGS_TARGET_COLUMNS:
    col_r2 = r2_score(y_test[col], y_pred_df[col])
    col_mae = mean_absolute_error(y_test[col], y_pred_df[col])
    console.print(f"  [cyan]{col:>10s}[/cyan] -> R2: {col_r2:.4f} | MAE: {col_mae:.4f}")

In [ ]:
fig, axes = plt.subplots(1, len(USGS_TARGET_COLUMNS), figsize=(7 * len(USGS_TARGET_COLUMNS), 6))

for ax, col in zip(axes, USGS_TARGET_COLUMNS):
    ax.scatter(
        y_test[col], y_pred_df[col], alpha=0.45, s=14,
        color=PLOT_COLORS["primary"], edgecolors="none",
    )
    lims = [y_test[col].min(), y_test[col].max()]
    ax.plot(lims, lims, "--", linewidth=1.8, color=PLOT_COLORS["accent"], label="Perfect prediction")
    ax.set_xlabel(f"Actual {col}")
    ax.set_ylabel(f"Predicted {col}")
    ax.set_title(f"Actual vs. Predicted {col}", fontweight="bold")
    ax.legend(frameon=False)
    ax.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "rf_actual_vs_predicted.png"), dpi=PLOT_DPI, bbox_inches="tight")
plt.show()

## 7 · Feature Importance

In [ ]:
importances = pd.Series(
    best_model.feature_importances_, index=USGS_FEATURE_COLUMNS
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x=importances.values, y=importances.index, hue=importances.index,
            palette="viridis", legend=False, edgecolor="none", ax=ax)
ax.set_title("Random Forest Feature Importance", fontweight="bold")
ax.set_xlabel("Importance")
ax.set_ylabel("")
ax.grid(alpha=0.3, axis="x")
fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "rf_feature_importance.png"), dpi=PLOT_DPI, bbox_inches="tight")
plt.show()

console.print(metrics_table("Feature Importance", importances.to_dict()))

**Interpretation:** Spatial features (`Latitude`, `Longitude`) typically dominate over `Timestamp` —
consistent with the geophysical expectation that *where* an event occurs (which fault system, which plate
boundary regime) is far more predictive of magnitude/depth than *when* it occurs.

## 8 · Persist the Trained Model

In [ ]:
section("Persist Model Artifact")

os.makedirs(MODELS_DIR, exist_ok=True)
model_path = os.path.join(MODELS_DIR, "random_forest_usgs.joblib")
joblib.dump(best_model, model_path)
console.print(f"[bold green]Model saved[/bold green] -> {model_path}")

---
## 9 · Result Summary

| Metric | Value |
|---|---|
| Best `n_estimators` | *(see GridSearchCV output above)* |
| R² Score | *(see Section 6 metrics table above)* |
| Train / Test Split | 80 / 20 |
| Cross-validation | 5-fold, scored on R² |

This Random Forest model establishes a strong tabular baseline for geospatial–temporal earthquake
characteristics, motivating the deeper signal-based approach explored in Notebooks 2 and 3, which work
directly with raw acoustic waveforms rather than tabular catalog records.
